# Veylo SLM Post-Processor: Unsloth QLoRA Fine-Tuning
### 0.5B Specialized Voice Intelligence (Hinglish, Fillers, Self-Corrections, Code)

In [ ]:
# 1. Install Unsloth and unsloth_zoo dependencies
!pip install unsloth unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets


In [ ]:
# 2. Fast Embedded 50,000 Multi-Domain Synthetic Dataset Generator
import json, random, os

SYSTEM_PROMPT = (
    "You are Veylo's intelligent speech post-processor. Clean the raw spoken transcription: "
    "remove filler words, fix self-corrections, punctuate properly, handle Hinglish naturally, "
    "and format code or emails cleanly. Output ONLY the polished text without explanation."
)

FILLERS_EN = ['um', 'uh', 'like', 'you know', 'basically', 'actually', 'I mean', 'sort of']
FILLERS_IN = ['matlab', 'dekho', 'bhai', 'yaar', 'arre', 'toh na', 'actually speaking']
NAMES_IN = ['Rahul', 'Priya', 'Amit', 'Sneha', 'Rohan', 'Ananya', 'Vikram', 'Pooja', 'Arjun', 'Neha']
NAMES_EN = ['Sarah', 'David', 'Alex', 'John', 'Emma', 'Michael', 'Sophia', 'Daniel']
TIMES = ['10:00 AM', '11:30 AM', '2:00 PM', '3:30 PM', '4:15 PM', '5:00 PM', '6:30 PM']
DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'tomorrow']

def inject_noise(text, hinglish=False):
    words = text.split()
    noisy = []
    f_list = (FILLERS_EN + FILLERS_IN) if hinglish else FILLERS_EN
    for w in words:
        if random.random() < 0.12:
            noisy.append(random.choice(f_list))
        if random.random() < 0.04 and len(w) > 2 and w[0].isalpha():
            noisy.append(f"{w[:2]}-{w}")
        else:
            noisy.append(w)
    raw = ' '.join(noisy).lower()
    for ch in ['.', ',', '?', '!', ':', ';']:
        raw = raw.replace(ch, '')
    return raw

def make_samples():
    data = []
    # Hinglish (15,000)
    h_tmpls = [
        ('kal subah {name} ke saath {time} pe meeting schedule kar do', 'Kal subah {name} ke saath {time} pe meeting schedule kar do.'),
        ('bhai sun ye wala PR review kar dena whenever you are free', 'Bhai sun, ye wala PR review kar dena whenever you are free.'),
        ('client ko quarterly roadmap ke regarding update bhej diya hai', 'Client ko quarterly roadmap ke regarding update bhej diya hai.'),
        ('production pe issue aa raha hai database latency bahut high ho gayi hai', 'Production pe issue aa raha hai, database latency bahut high ho gayi hai.'),
        ('chalo done hai kal {day} ko connect karte hain shaam ko {time}', 'Chalo done hai, kal {day} ko shaam {time} pe connect karte hain.')
    ]
    for _ in range(15000):
        r_t, c_t = random.choice(h_tmpls)
        n, t, d = random.choice(NAMES_IN), random.choice(TIMES), random.choice(DAYS)
        data.append((inject_noise(r_t.format(name=n, time=t, day=d), True), c_t.format(name=n, time=t, day=d)))

    # Self-Corrections (12,500)
    c_tmpls = [
        ('send this document to {n1} wait no send it to {n2}', 'Send this document to {n2}.'),
        ('schedule the demo for {t1} actually make that {t2}', 'Schedule the demo for {t2}.'),
        ('we need three servers sorry we need four servers for the cluster', 'We need four servers for the cluster.'),
        ('call {n1} wait message him on slack instead', 'Message {n1} on Slack instead.'),
        ('deploy to staging first no wait push directly to production', 'Push directly to production.')
    ]
    for _ in range(12500):
        r_t, c_t = random.choice(c_tmpls)
        n1, n2 = random.sample(NAMES_EN + NAMES_IN, 2)
        t1, t2 = random.sample(TIMES, 2)
        data.append((inject_noise(r_t.format(n1=n1, n2=n2, t1=t1, t2=t2), False), c_t.format(n1=n1, n2=n2, t1=t1, t2=t2)))

    # Code (10,000)
    cd_tmpls = [
        ('create a function named get user by id taking user id as string and returning user', 'def get_user_by_id(user_id: str) -> User:'),
        ('const is loading set is loading equals use state false', 'const [isLoading, setIsLoading] = useState(false);'),
        ('select id email and created at from users table where active is true', 'SELECT id, email, created_at FROM users WHERE active = true;'),
        ('git checkout minus b feature slash authentication', 'git checkout -b feature/authentication'),
        ('run cargo build release in the root directory', 'cargo build --release')
    ]
    for _ in range(10000):
        r_t, c_t = random.choice(cd_tmpls)
        data.append((inject_noise(r_t, False), c_t))

    # Email (7,500)
    em_tmpls = [
        ('hi {name} new line please find attached the weekly sprint report thanks and regards {sender}', 'Hi {name},\n\nPlease find attached the weekly sprint report.\n\nThanks and regards,\n{sender}'),
        ('dear team new line we are excited to announce our release tomorrow at {time} best regards {sender}', 'Dear Team,\n\nWe are excited to announce our release tomorrow at {time}.\n\nBest regards,\n{sender}')
    ]
    for _ in range(7500):
        r_t, c_t = random.choice(em_tmpls)
        n, s, t = random.choice(NAMES_EN), random.choice(NAMES_IN), random.choice(TIMES)
        data.append((inject_noise(r_t.format(name=n, sender=s, time=t), False), c_t.format(name=n, sender=s, time=t)))

    # Fillers (5,000)
    fl_tmpls = [
        ('we should probably inspect the redis cache before scaling up the instances', 'We should inspect the Redis cache before scaling up the instances.'),
        ('the latency dropped from one hundred milliseconds to fifteen milliseconds', 'The latency dropped from 100ms to 15ms.'),
        ('make sure to write unit tests for the authentication module', 'Make sure to write unit tests for the authentication module.')
    ]
    for _ in range(5000):
        r_t, c_t = random.choice(fl_tmpls)
        data.append((inject_noise(f'so like basically um {r_t} you know', False), c_t))

    random.seed(42)
    random.shuffle(data)
    out_path = '/kaggle/working/veylo_train_50k.jsonl'
    with open(out_path, 'w', encoding='utf-8') as f:
        for raw, target in data:
            rec = {'messages': [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': raw}, {'role': 'assistant', 'content': target}]}
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'Generated {len(data)} training samples at {out_path} ({os.path.getsize(out_path)/1024/1024:.2f} MB)!')

make_samples()


In [ ]:
# 3. Load Base Model with Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None # Auto detection (FP16 on Tesla T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


In [ ]:
# 4. Attach QLoRA Adaptors
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


In [ ]:
# 5. Format Dataset with ChatML Template
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

from datasets import load_dataset
dataset = load_dataset("json", data_files="/kaggle/working/veylo_train_50k.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)


In [ ]:
# 6. Train with SFTTrainer and Completion-Only Loss Masking
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        max_steps = 1500, # ~25-30 min on dual T4
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Mask user prompt from loss calculation
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

print('Starting Veylo SLM training on GPU...')
trainer_stats = trainer.train()
print('Training complete! Loss stats:', trainer_stats)


In [ ]:
# 7. Live Validation Tests
FastLanguageModel.for_inference(model)

test_cases = [
    "uh bhai sun meeting schedule kar do kal shaam ko 5 baje",
    "send this email to alex wait no send it to david instead",
    "you know create a function named calculate total taking price and tax as floats"
]

print('=== VEYLO SLM INFERENCE VALIDATION ===')
for raw in test_cases:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": raw}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True)
    clean_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'RAW:   {raw}')
    print(f'CLEAN: {clean_text}')
    print('-' * 50)


In [ ]:
# 8. Export directly to GGUF Q4_K_M for Rust Inference
import os
print('Exporting fine-tuned Veylo model to GGUF Q4_K_M...')
model.save_pretrained_gguf('/kaggle/working/veylo-qwen-0.5b', tokenizer, quantization_method = 'q4_k_m')
print('GGUF export finished! Files in /kaggle/working:')
for f in os.listdir('/kaggle/working'):
    if f.endswith('.gguf'):
        sz = os.path.getsize(os.path.join('/kaggle/working', f)) / (1024 * 1024)
        print(f'  -> {f} ({sz:.2f} MB)')
